# 02 - Silver Layer: Cleaning & Conformance

## What this notebook does
Transforms bronze (raw, as-landed, append-only) tables into clean, conformed
silver tables, one per Salesforce object:
- Picks the latest known version of each record (by `SystemModstamp`), since
  bronze can hold multiple landings of the same `Id` across different
  incremental extraction runs
- Drops soft-deleted records (`IsDeleted = true`)
- Casts date strings (`CreatedDate`, `CloseDate`, `SystemModstamp`) to real
  DATE/TIMESTAMP types, since Autoloader's JSON inference left them as strings
- Renames a few fields to consistent snake_case names for downstream use

## Why full overwrite, not incremental MERGE
Bronze already retains full history via Autoloader, and this data volume is
small (roughly 1,000 records total), so silver is recomputed from bronze in
full on every run - the same choice Project 1 made for `silver_beneficiary`.
True incremental MERGE upsert logic is deliberately saved for the Credit Card
project, where it actually matters at scale.

## Tables created
- `main.sales_revenue_analytics.silver_account`
- `main.sales_revenue_analytics.silver_contact`
- `main.sales_revenue_analytics.silver_lead`
- `main.sales_revenue_analytics.silver_opportunity`
- `main.sales_revenue_analytics.silver_campaign`

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, to_date, to_timestamp, when

bronze_opportunity = spark.table("main.sales_revenue_analytics.bronze_opportunity")

# Bronze can hold several landings of the same Id across different extraction
# runs (an original insert, then later updates). Keep only the newest one per Id.
latest_per_id = Window.partitionBy("Id").orderBy(col("SystemModstamp").desc())

silver_opportunity = (
    bronze_opportunity
    .withColumn("_rn", row_number().over(latest_per_id))
    .filter(col("_rn") == 1)
    .filter(col("IsDeleted") == False)  # noqa: E712 - drop soft-deleted records
    .withColumn("close_date", to_date("CloseDate", "yyyy-MM-dd"))
    .withColumn("created_at", to_timestamp("CreatedDate", "yyyy-MM-dd'T'HH:mm:ss.SSSXX"))
    .withColumn("last_modified_at", to_timestamp("SystemModstamp", "yyyy-MM-dd'T'HH:mm:ss.SSSXX"))
    .withColumn("is_closed", col("StageName").isin("Closed Won", "Closed Lost"))
    .withColumn("is_won", col("StageName") == "Closed Won")
    .withColumnRenamed("Id", "opportunity_id")
    .withColumnRenamed("Name", "name")
    .withColumnRenamed("AccountId", "account_id")
    .withColumnRenamed("StageName", "stage")
    .withColumnRenamed("Amount", "amount")
    .withColumnRenamed("LeadSource", "lead_source")
    .withColumnRenamed("Sales_Rep__c", "sales_rep")
    .select(
        "opportunity_id", "name", "account_id", "stage", "is_closed", "is_won",
        "amount", "close_date", "lead_source", "sales_rep",
        "created_at", "last_modified_at",
    )
)

(
    silver_opportunity.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.sales_revenue_analytics.silver_opportunity")
)

print(f"silver_opportunity: {silver_opportunity.count()} rows")

silver_opportunity: 403 rows


In [0]:
bronze_account = spark.table("main.sales_revenue_analytics.bronze_account")

latest_per_id = Window.partitionBy("Id").orderBy(col("SystemModstamp").desc())

silver_account = (
    bronze_account
    .withColumn("_rn", row_number().over(latest_per_id))
    .filter(col("_rn") == 1)
    .filter(col("IsDeleted") == False)  # noqa: E712
    .withColumn("created_at", to_timestamp("CreatedDate", "yyyy-MM-dd'T'HH:mm:ss.SSSXX"))
    .withColumnRenamed("Id", "account_id")
    .withColumnRenamed("Name", "name")
    .withColumnRenamed("Industry", "industry")
    .select("account_id", "name", "industry", "created_at")
)

(
    silver_account.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.sales_revenue_analytics.silver_account")
)

print(f"silver_account: {silver_account.count()} rows")

silver_account: 68 rows


In [0]:
bronze_contact = spark.table("main.sales_revenue_analytics.bronze_contact")

latest_per_id = Window.partitionBy("Id").orderBy(col("SystemModstamp").desc())

silver_contact = (
    bronze_contact
    .withColumn("_rn", row_number().over(latest_per_id))
    .filter(col("_rn") == 1)
    .filter(col("IsDeleted") == False)  # noqa: E712
    .withColumn("created_at", to_timestamp("CreatedDate", "yyyy-MM-dd'T'HH:mm:ss.SSSXX"))
    .withColumnRenamed("Id", "contact_id")
    .withColumnRenamed("AccountId", "account_id")
    .withColumnRenamed("Name", "name")
    .select("contact_id", "account_id", "name", "created_at")
)

(
    silver_contact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.sales_revenue_analytics.silver_contact")
)

print(f"silver_contact: {silver_contact.count()} rows")

silver_contact: 20 rows


In [0]:
bronze_lead = spark.table("main.sales_revenue_analytics.bronze_lead")

latest_per_id = Window.partitionBy("Id").orderBy(col("SystemModstamp").desc())

silver_lead = (
    bronze_lead
    .withColumn("_rn", row_number().over(latest_per_id))
    .filter(col("_rn") == 1)
    .filter(col("IsDeleted") == False)  # noqa: E712
    .withColumn("created_at", to_timestamp("CreatedDate", "yyyy-MM-dd'T'HH:mm:ss.SSSXX"))
    .withColumnRenamed("Id", "lead_id")
    .withColumnRenamed("FirstName", "first_name")
    .withColumnRenamed("LastName", "last_name")
    .withColumnRenamed("Company", "company")
    .withColumnRenamed("Email", "email")
    .withColumnRenamed("Status", "status")
    .withColumnRenamed("LeadSource", "lead_source")
    .select("lead_id", "first_name", "last_name", "company", "email",
           "status", "lead_source", "created_at")
)

(
    silver_lead.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.sales_revenue_analytics.silver_lead")
)

print(f"silver_lead: {silver_lead.count()} rows")

silver_lead: 503 rows


In [0]:
bronze_campaign = spark.table("main.sales_revenue_analytics.bronze_campaign")

latest_per_id = Window.partitionBy("Id").orderBy(col("SystemModstamp").desc())

silver_campaign = (
    bronze_campaign
    .withColumn("_rn", row_number().over(latest_per_id))
    .filter(col("_rn") == 1)
    .filter(col("IsDeleted") == False)  # noqa: E712
    .withColumn("created_at", to_timestamp("CreatedDate", "yyyy-MM-dd'T'HH:mm:ss.SSSXX"))
    .withColumnRenamed("Id", "campaign_id")
    .withColumnRenamed("Name", "name")
    .select("campaign_id", "name", "created_at")
)

(
    silver_campaign.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.sales_revenue_analytics.silver_campaign")
)

print(f"silver_campaign: {silver_campaign.count()} rows")

silver_campaign: 4 rows


In [0]:
%sql
SELECT 'silver_account' AS table_name, COUNT(*) AS row_count FROM main.sales_revenue_analytics.silver_account
UNION ALL
SELECT 'silver_contact', COUNT(*) FROM main.sales_revenue_analytics.silver_contact
UNION ALL
SELECT 'silver_lead', COUNT(*) FROM main.sales_revenue_analytics.silver_lead
UNION ALL
SELECT 'silver_opportunity', COUNT(*) FROM main.sales_revenue_analytics.silver_opportunity
UNION ALL
SELECT 'silver_campaign', COUNT(*) FROM main.sales_revenue_analytics.silver_campaign;

table_name,row_count
silver_account,68
silver_contact,20
silver_lead,503
silver_opportunity,403
silver_campaign,4


In [0]:
%sql
SELECT COUNT(*) AS orphaned_opportunities
FROM main.sales_revenue_analytics.silver_opportunity o
LEFT JOIN main.sales_revenue_analytics.silver_account a ON o.account_id = a.account_id
WHERE a.account_id IS NULL;

orphaned_opportunities
0
